# Pré-processamento de Dados para Aprendizado de Máquina

## Configuração inicial

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

np.random.seed(42)          # reprodutibilidade

In [2]:
# ── Configurações de Plotagem ────────────────────────────────────────────────
plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams['mathtext.rm'] = "Arial"
plt.rcParams['mathtext.it'] = "Arial:italic"
plt.rcParams['mathtext.bf'] = "Arial:bold"
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Integração de Dados

Integrar é reunir dados de **múltiplas fontes** em um conjunto único e coerente.
A operação central é a **junção** (*join*): combinar linhas de duas tabelas
sempre que elas concordam no valor de uma **chave comum**.

Vamos usar duas tabelas pequenas para enxergar o efeito de cada tipo de junção.

In [3]:
clientes = pd.DataFrame({
    "id":     [1, 2, 3],
    "nome":   ["Ana", "João", "Bia"],
    "cidade": ["Natal", "Mossoró", "Pau dos Ferros"],
})

compras = pd.DataFrame({
    "id":      [1, 1, 2, 4],
    "produto": ["livro", "caneta", "mochila", "fone"],
    "valor":   [80, 5, 150, 60],
})

print("Clientes:"); display(clientes)
print("Compras:");  display(compras)

Clientes:


,id,nome,cidade
0,1,Ana,Natal
1,2,João,Mossoró
2,3,Bia,Pau dos Ferros


Compras:


,id,produto,valor
0,1,livro,80
1,1,caneta,5
2,2,mochila,150
3,4,fone,60


Repare em dois casos propositais:

- o cliente `id=3` (Bia) **não tem compra**;
- a compra `id=4` (fone) **não tem cliente** correspondente.

É o destino desses dois registros que diferencia os tipos de junção.

In [4]:
for tipo in ["inner", "left", "right", "outer"]:
    print(f"Tipo ='{tipo}'")
    display(pd.merge(clientes, compras, on="id", how=tipo))

Tipo ='inner'


,id,nome,cidade,produto,valor
0,1,Ana,Natal,livro,80
1,1,Ana,Natal,caneta,5
2,2,João,Mossoró,mochila,150


Tipo ='left'


,id,nome,cidade,produto,valor
0,1,Ana,Natal,livro,80.0
1,1,Ana,Natal,caneta,5.0
2,2,João,Mossoró,mochila,150.0
3,3,Bia,Pau dos Ferros,NaN,NaN


Tipo ='right'


,id,nome,cidade,produto,valor
0,1,Ana,Natal,livro,80
1,1,Ana,Natal,caneta,5
2,2,João,Mossoró,mochila,150
3,4,NaN,NaN,fone,60


Tipo ='outer'


,id,nome,cidade,produto,valor
0,1,Ana,Natal,livro,80.0
1,1,Ana,Natal,caneta,5.0
2,2,João,Mossoró,mochila,150.0
3,3,Bia,Pau dos Ferros,NaN,NaN
4,4,NaN,NaN,fone,60.0


In [5]:
a = pd.merge(clientes, compras, on="id", how="left")
b = pd.merge(compras, clientes, on="id", how="left")

print("clientes ⟕ compras:")
display(a)
print("compras ⟕ clientes")
display(b)

clientes ⟕ compras:


,id,nome,cidade,produto,valor
0,1,Ana,Natal,livro,80.0
1,1,Ana,Natal,caneta,5.0
2,2,João,Mossoró,mochila,150.0
3,3,Bia,Pau dos Ferros,NaN,NaN


compras ⟕ clientes


,id,produto,valor,nome,cidade
0,1,livro,80,Ana,Natal
1,1,caneta,5,Ana,Natal
2,2,mochila,150,João,Mossoró
3,4,fone,60,NaN,NaN


## 2. Limpeza de Dados

Vamos construir um dataset propositalmente "sujo", a partir do **Iris**, contendo os quatro problemas clássicos: dados **incompletos**, **inconsistentes**, **redundantes** e **com ruído**.

In [6]:
from sklearn.datasets import load_iris

dados = load_iris(as_frame=True)
iris = dados.frame.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)":  "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)":  "petal_width",
})
iris["species"] = pd.Categorical.from_codes(dados.target, dados.target_names)
iris = iris.drop(columns="target")

print(iris.shape)
iris.head()

(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [7]:
# --- criando os problemas de propósito ---
sujo = iris.copy()

# (a) INCOMPLETOS: valores faltantes
sujo.loc[[3, 17, 42], "sepal_length"] = np.nan
sujo.loc[[8, 25], "species"] = np.nan

# (b) INCONSISTENTES: valores impossíveis (medida negativa)
sujo.loc[10, "petal_width"] = -1.5

# (c) REDUNDANTES: duplicar linhas existentes
sujo = pd.concat([sujo, sujo.iloc[[0, 1]]], ignore_index=True)

# (d) RUÍDO: outlier grosseiro (erro de digitação: 50 em vez de 5.0)
sujo.loc[60, "sepal_length"] = 50.0

#### Valores Faltantes

In [8]:
display(sujo.iloc[1:4]) 

,sepal_length,sepal_width,petal_length,petal_width,species
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,NaN,3.1,1.5,0.2,setosa


In [9]:
display(sujo.iloc[5:10]) 

,sepal_length,sepal_width,petal_length,petal_width,species
5,5.4,3.9,1.7,0.4,setosa
6,4.6,3.4,1.4,0.3,setosa
7,5.0,3.4,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,NaN
9,4.9,3.1,1.5,0.1,setosa


#### Dados Inconsistentes

In [10]:
display(sujo.iloc[8:11]) 

,sepal_length,sepal_width,petal_length,petal_width,species
8,4.4,2.9,1.4,0.2,NaN
9,4.9,3.1,1.5,0.1,setosa
10,5.4,3.7,1.5,-1.5,setosa


#### Dados com Ruído

In [11]:
display(sujo.iloc[58:61]) 

,sepal_length,sepal_width,petal_length,petal_width,species
58,6.6,2.9,4.6,1.3,versicolor
59,5.2,2.7,3.9,1.4,versicolor
60,50.0,2.0,3.5,1.0,versicolor


In [15]:
display(sujo.iloc[0:2],sujo.iloc[150:152]) 

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa


,sepal_length,sepal_width,petal_length,petal_width,species
150,5.1,3.5,1.4,0.2,setosa
151,4.9,3.0,1.4,0.2,setosa


### 2.1 - Dados Incompletos

Primeiro **diagnosticamos** onde estão os faltantes; depois decidimos o tratamento: remover ou **imputar**.

In [16]:
print("Faltantes por coluna:")
print(sujo.isna().sum())

Faltantes por coluna:
sepal_length    3
sepal_width     0
petal_length    0
petal_width     0
species         2
dtype: int64


In [17]:
print("Linhas com algum faltante:")
display(sujo[sujo.isna().any(axis=1)])

Linhas com algum faltante:


,sepal_length,sepal_width,petal_length,petal_width,species
3,NaN,3.1,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,NaN
17,NaN,3.5,1.4,0.3,setosa
25,5.0,3.0,1.6,0.2,NaN
42,NaN,3.2,1.3,0.2,setosa


In [18]:
limpo_etapa1 = sujo.copy()

# Imputação: mediana para numérico (robusta a outliers), moda para categórico
mediana = sujo["sepal_length"].median()
moda    = sujo["species"].mode()[0]   #Pega a primeira moda

limpo_etapa1["sepal_length"] = limpo_etapa1["sepal_length"].fillna(mediana)
limpo_etapa1["species"]      = limpo_etapa1["species"].fillna(moda)

display(limpo_etapa1[sujo.isna().any(axis=1)])

,sepal_length,sepal_width,petal_length,petal_width,species
3,5.8,3.1,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,setosa
17,5.8,3.5,1.4,0.3,setosa
25,5.0,3.0,1.6,0.2,setosa
42,5.8,3.2,1.3,0.2,setosa


In [22]:
print("Linhas com algum faltante:")
print(limpo_etapa1.isna().sum())

Linhas com algum faltante:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


O scikit-learn oferece o `SimpleImputer`, que é preferível dentro de um
`Pipeline` — ele guarda os valores aprendidos no **treino** e os reaplica no
**teste**, evitando *data leakage*.

In [23]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median")
X_num = sujo[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
X_imp = imp.fit_transform(X_num)

In [24]:
X_imp_df = pd.DataFrame(X_imp, columns=X_num.columns, index=X_num.index)
display(X_imp_df[sujo.isna().any(axis=1)])

,sepal_length,sepal_width,petal_length,petal_width
3,5.8,3.1,1.5,0.2
8,4.4,2.9,1.4,0.2
17,5.8,3.5,1.4,0.3
25,5.0,3.0,1.6,0.2
42,5.8,3.2,1.3,0.2


### 2.2 - Dados Inconsistentes

São valores que **violam regras do domínio** — aqui, uma medida física negativa. O tratamento passa por validação por faixas.

In [25]:
limpo_etapa2 = limpo_etapa1.copy()

In [26]:
# 1) Identifica os inconsistentes (sem alterar nada ainda)
num_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
inconsistentes = limpo_etapa1[num_cols] <= 0
linhas_inconsistentes = inconsistentes.any(axis=1)

print("Valores inconsistentes por coluna:")
print(inconsistentes.sum())
print("\nLinhas afetadas:", linhas_inconsistentes.sum())
display(limpo_etapa1.loc[linhas_inconsistentes])

Valores inconsistentes por coluna:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     1
dtype: int64

Linhas afetadas: 1


,sepal_length,sepal_width,petal_length,petal_width,species
10,5.4,3.7,1.5,-1.5,setosa


In [27]:
# Tratamento: marcar como faltante e imputar
limpo_etapa2[num_cols] = limpo_etapa1[num_cols].mask(inconsistentes)
limpo_etapa2[num_cols] = limpo_etapa2[num_cols].fillna(limpo_etapa2[num_cols].median())

print("Após correção:", (limpo_etapa2[num_cols] <= 0).any(axis=1).sum())
display(limpo_etapa2.loc[linhas_inconsistentes])          # <- as MESMAS linhas, já corrigidas

Após correção: 0


,sepal_length,sepal_width,petal_length,petal_width,species
10,5.4,3.7,1.5,1.3,setosa


### 2.3 - Dados redundantes

Duas formas de redundância: **linhas duplicadas** e **atributos altamente correlacionados** (um derivável do outro).

In [ ]:
limpo_etapa3 = limpo_etapa2.copy()

In [ ]:
print("Linhas duplicadas:", limpo_etapa2.duplicated().sum())

dups = limpo_etapa2[limpo_etapa2.duplicated(keep=False)]
display(dups.sort_values(by=list(limpo_etapa2.columns)))

In [ ]:
limpo_etapa3 = limpo_etapa2.drop_duplicates().reset_index(drop=True)
print("Formato após remover duplicatas:", limpo_etapa3.shape)

In [ ]:
# Redundância entre ATRIBUTOS: correlação alta
corr = limpo_etapa3[num_cols].corr()
display(corr.round(2))

# Pares com |correlação| > 0.9 (candidatos a remoção)
alta = [(i, j, corr.loc[i, j])
        for i in num_cols for j in num_cols
        if i < j and abs(corr.loc[i, j]) > 0.9]
print("Pares muito correlacionados:", alta)

### 2.4 - Dados com ruído — detecção por IQR

Regra do boxplot: é *outlier* todo valor fora de
$[\,Q_1 - 1{,}5\,IQR,\ Q_3 + 1{,}5\,IQR\,]$, com $IQR = Q_3 - Q_1$.

É um método **univariado** (analisa um atributo por vez) e **robusto**
(não assume distribuição normal).

In [ ]:
limpo_etapa4 = limpo_etapa3.copy()

In [ ]:
def outliers_iqr(serie):
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return (serie < lim_inf) | (serie > lim_sup), lim_inf, lim_sup

In [ ]:
mask, li, ls = outliers_iqr(limpo_etapa3["sepal_length"])

print(f"Limites aceitáveis: [{li:.2f}, {ls:.2f}]")
print("Outliers detectados:", mask.sum())

display(limpo_etapa3.loc[mask, ["sepal_length", "species"]])

In [ ]:
mascaras = {}

for col in num_cols:
    mask, li, ls = outliers_iqr(limpo_etapa3[col])
    mascaras[col] = mask
    print(f"{col:15s} limites: [{li:6.2f}, {ls:6.2f}]  ->  {mask.sum()} outlier(s)")
    if mask.any():
        display(limpo_etapa3.loc[mask, [col, "species"]])

# Linhas com outlier em ALGUMA feature
mask_qualquer = pd.DataFrame(mascaras).any(axis=1)
print("Linhas com pelo menos um outlier:", mask_qualquer.sum())
display(limpo_etapa3.loc[mask_qualquer])

In [ ]:
limpo_etapa3[num_cols].plot(kind="box", figsize=(8, 5))
plt.ylabel("Tamanho (cm)")
plt.title("Boxplots dos atributos")
plt.grid(False)
plt.show()

In [ ]:
limpo_etapa4 = limpo_etapa3[~mask_qualquer].reset_index(drop=True)

print("Antes :", limpo_etapa3.shape)
print("Depois:", limpo_etapa4.shape)
print("Removidas:", mask_qualquer.sum(), "linha(s)")

In [ ]:
limpo_etapa4[num_cols].plot(kind="box", figsize=(8, 5))
plt.ylabel("Tamanho (cm)")
plt.title("Boxplots dos atributos")
plt.grid(False)
plt.show()

### 2.5 - Dados com ruído — detecção por LOF

O **Local Outlier Factor** é **multivariado** e baseado em **densidade**: compara a densidade local de cada ponto com a de seus $k$ vizinhos.

- $\mathrm{LOF} \approx 1$ → densidade semelhante à dos vizinhos (normal)
- $\mathrm{LOF} \gg 1$ → ponto isolado → provável *outlier*

Diferente do IQR, ele detecta outliers **locais**: pontos que são normais globalmente, mas anômalos em relação à sua vizinhança.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

X = limpo_etapa3[num_cols].values

lof = LocalOutlierFactor(n_neighbors=20)
rotulo = lof.fit_predict(X)          # -1 = outlier, 1 = normal

# negative_outlier_factor_ é o simétrico do LOF;
escore = -lof.negative_outlier_factor_

print("Outliers detectados pelo LOF:", (rotulo == -1).sum())
print("Escore LOF — mín: %.2f | mediana: %.2f | máx: %.2f"
      % (escore.min(), np.median(escore), escore.max()))

In [ ]:
limpo_etapa4 = limpo_etapa3[rotulo == 1].reset_index(drop=True)

print("Antes :", limpo_etapa3.shape)
print("Depois:", limpo_etapa4.shape)
print("Removidas:", (rotulo == -1).sum(), "linha(s)")

**Comparando as duas abordagens:** o IQR olha cada atributo isoladamente,
enquanto o LOF considera a posição do ponto no espaço multidimensional. Um ponto
pode ter valores perfeitamente normais em *cada* atributo e ainda assim ser uma
**combinação anômala** — só o LOF detecta esse caso.

In [ ]:
# Um ponto com valores individualmente normais, mas COMBINAÇÃO anômala
anomalo = pd.DataFrame([{
    "sepal_length": 5.0,   # típico de setosa
    "sepal_width":  3.4,   # típico de setosa
    "petal_length": 5.5,   # típico de virginica (!)
    "petal_width":  2.0,   # típico de virginica (!)
}])

teste = pd.concat([limpo_etapa3[num_cols], anomalo], ignore_index=True)

# IQR não acusa: cada valor está dentro da faixa do seu atributo
Q1, Q3 = teste.quantile(0.25), teste.quantile(0.75)
iqr_flag = ((anomalo < Q1 - 1.5 * (Q3 - Q1)) |
            (anomalo > Q3 + 1.5 * (Q3 - Q1))).any(axis=1).iloc[0]

# LOF acusa: a combinação não existe em nenhum grupo
lof2 = LocalOutlierFactor(n_neighbors=20)
lof_flag = lof2.fit_predict(teste.values)[-1] == -1

print("IQR marcou como outlier?", iqr_flag)
print("LOF marcou como outlier?", lof_flag)

## 3. Transformação de Dados

Adequar os atributos ao que o algoritmo espera:

- **simbólico → numérico**: algoritmos numéricos exigem números;
- **numérico → simbólico**: discretização em faixas;
- **transformação de numéricos**: normalização e padronização.

### 3.1 - Simbólico → Numérico

Duas codificações, e a escolha depende do tipo do atributo:

- **Ordinal / Label**: para categorias **com ordem** natural;
- **One-Hot**: para categorias **nominais** (sem ordem).

Usar *label encoding* em dado nominal cria uma **ordem falsa** que o modelo interpretará como informação real.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder

# (a) ORDINAL — há ordem natural
escolaridade = pd.DataFrame({"nivel": ["Médio", "Fundamental", "Superior", "Médio"]})
ordem = [["Fundamental", "Médio", "Superior"]]

oe = OrdinalEncoder(categories=ordem)
escolaridade["codigo"] = oe.fit_transform(escolaridade[["nivel"]]).astype(int)
display(escolaridade)

In [ ]:
escolaridade = escolaridade.drop(columns="nivel")
display(escolaridade)

In [ ]:
# (b) ONE-HOT — categorias nominais (species do Iris)
oh = OneHotEncoder(sparse_output=False)
especies_ohe = oh.fit_transform(limpo_etapa4[["species"]])

df_ohe = pd.DataFrame(especies_ohe, columns=oh.get_feature_names_out(["species"]))
df_ohe_concat = pd.concat([limpo_etapa4[["sepal_length", "sepal_width", "petal_length", "petal_width"]], df_ohe], axis=1)
display(df_ohe_concat)

### 3.2 Numérico → Simbólico (discretização)

Agrupar valores contínuos em **faixas** (*bins*). Duas estratégias usuais:

- `uniform` — faixas de **largura igual**;
- `quantile` — faixas com **frequência igual** (mesmo nº de amostras cada).

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

col = limpo_etapa4[["petal_length"]]

for estrategia in ["uniform", "quantile"]:
    disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy=estrategia)
    faixas = disc.fit_transform(col).astype(int).ravel()
    contagem = pd.Series(faixas).value_counts().sort_index()
    print(f"strategy='{estrategia}':")
    print("  cortes  :", np.round(disc.bin_edges_[0], 2))
    print("  amostras por faixa:", contagem.tolist())

## 3.3 Normalização e padronização

Muitos algoritmos (k-NN, SVM, redes neurais, PCA) são sensíveis à **escala**.

- **Normalização** (min-max): $x' = \dfrac{x - x_{\min}}{x_{\max} - x_{\min}} \in [0, 1]$
- **Padronização** (z-score): $x' = \dfrac{x - \mu}{\sigma}$, com média 0 e desvio 1
- **Robusta**: usa mediana e IQR — menos sensível a outliers

Usamos o **Wine**, cujos atributos têm escalas muito diferentes.

In [ ]:
from sklearn.datasets import load_wine

vinho = load_wine(as_frame=True)
Xw, yw = vinho.data, vinho.target

# Escalas bem distintas entre os atributos
display(Xw[["proline", "magnesium", "malic_acid", "hue"]].describe().round(2))

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

escaladores = {
    "Original":     None,
    "MinMax [0,1]": MinMaxScaler(),
    "Z-score":      StandardScaler(),
    "Robusta":      RobustScaler(),
}

for nome, esc in escaladores.items():
    Xt = Xw.values if esc is None else esc.fit_transform(Xw)
    print(f"{nome:14s} | média: {Xt.mean():8.2f} | desvio: {Xt.std():7.2f} "
          f"| mín: {Xt.min():7.2f} | máx: {Xt.max():8.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].boxplot(Xw[["magnesium", "proline", "hue"]].values,
              tick_labels=["magnesium", "proline", "hue"])
ax[0].set_title("Antes: escalas incompatíveis")

Xw_pad = StandardScaler().fit_transform(Xw)
idx = [Xw.columns.get_loc(c) for c in ["magnesium", "proline", "hue"]]
ax[1].boxplot(Xw_pad[:, idx], tick_labels=["magnesium", "proline", "hue"])
ax[1].set_title("Depois: padronizado (z-score)")

for a in ax:
    a.grid(False)
plt.tight_layout()
plt.show()

> **Atenção ao vazamento (*data leakage*)**: os parâmetros do escalador
> ($x_{\min}$, $x_{\max}$, $\mu$, $\sigma$) devem ser aprendidos **apenas no treino**
> (`fit`) e depois aplicados ao teste (`transform`). Fazer `fit` no dataset
> completo contamina a validação.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(Xw, yw, test_size=0.3,
                                          stratify=yw, random_state=42)

esc = StandardScaler()
X_tr_pad = esc.fit_transform(X_tr)   # fit + transform no TREINO
X_te_pad = esc.transform(X_te)       # apenas transform no TESTE

print("Treino — média: %.3f | desvio: %.3f" % (X_tr_pad.mean(), X_tr_pad.std()))
print("Teste  — média: %.3f | desvio: %.3f  (não é exatamente 0 e 1: correto!)"
      % (X_te_pad.mean(), X_te_pad.std()))

<a id="4"></a>
## 4. Redução de Dimensionalidade — PCA

O PCA projeta os dados em novas direções ortogonais (**componentes principais**)
que **maximizam a variância**.

Passos do algoritmo:

1. centralizar: $\tilde{X} = X - \bar{x}$
2. covariância: $\Sigma = \frac{1}{n-1}\tilde{X}^{\top}\tilde{X}$
3. autovalores/autovetores: $\Sigma v_j = \lambda_j v_j$
4. ordenar por $\lambda_j$ e selecionar os $k$ maiores → $W_k$
5. projetar: $Z = \tilde{X} W_k$

Começamos implementando **na mão** para ver cada passo.

In [ ]:
# PCA passo a passo com numpy (Wine padronizado)
Xp = StandardScaler().fit_transform(Xw)
n = Xp.shape[0]

# 1) centralizar (já está centrado após o z-score, mas explicitamos o passo)
x_barra = Xp.mean(axis=0)
X_til = Xp - x_barra
print("Vetor média (5 primeiros):", np.round(x_barra[:5], 4))

# 2) matriz de covariância
Sigma = (X_til.T @ X_til) / (n - 1)
print("Formato de Sigma:", Sigma.shape)

# 3) autovalores e autovetores
lambdas, V = np.linalg.eigh(Sigma)

# 4) ordenar em ordem decrescente de autovalor
ordem = np.argsort(lambdas)[::-1]
lambdas, V = lambdas[ordem], V[:, ordem]
print("5 maiores autovalores:", np.round(lambdas[:5], 3))

# 5) projetar nos k = 2 primeiros componentes
W_k = V[:, :2]
print(W_k)
Z = X_til @ W_k
print("Formato projetado:", Z.shape)

var_explicada = lambdas / lambdas.sum()
print("Variância explicada (PC1, PC2): %.1f%%, %.1f%%"
      % (var_explicada[0] * 100, var_explicada[1] * 100))

In [ ]:
# Mesmo resultado com o scikit-learn
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
Z_sk = pca.fit_transform(Xp)

print("Variância explicada:", np.round(pca.explained_variance_ratio_, 4))
print("Acumulada: %.1f%%" % (pca.explained_variance_ratio_.sum() * 100))

# Os sinais podem diferir (autovetor é definido a menos de sinal)
print("\nCoincide com a implementação manual?",
      np.allclose(np.abs(Z), np.abs(Z_sk)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) projeção nas duas primeiras componentes
for classe, nome in enumerate(vinho.target_names):
    m = yw == classe
    ax[0].scatter(Z_sk[m, 0], Z_sk[m, 1], label=nome, s=25)
ax[0].set_xlabel("PC1 (%.1f%%)" % (pca.explained_variance_ratio_[0] * 100))
ax[0].set_ylabel("PC2 (%.1f%%)" % (pca.explained_variance_ratio_[1] * 100))
ax[0].set_title("Wine projetado em 2 componentes")
ax[0].legend()

# (b) variância explicada acumulada (scree plot)
pca_full = PCA().fit(Xp)
acum = np.cumsum(pca_full.explained_variance_ratio_)
ax[1].plot(range(1, len(acum) + 1), acum, marker="o")
ax[1].axhline(0.95, color="r", linestyle="--", label="95%")
ax[1].set_xlabel("Número de componentes")
ax[1].set_ylabel("Variância explicada acumulada")
ax[1].set_title("Quantas componentes manter?")
ax[1].legend()

for a in ax:
    a.grid(False)
plt.tight_layout()
plt.show()

k95 = np.argmax(acum >= 0.95) + 1
print(f"Componentes necessárias para 95%% da variância: {k95} (de {Xp.shape[1]})")

**Por que padronizar antes do PCA?** O PCA maximiza variância — sem
padronizar, atributos de maior magnitude dominam as componentes. No Wine, o
atributo `proline` (centenas) esmagaria `hue` (unidades).

In [ ]:
pca_sem = PCA(n_components=2).fit(Xw)      # SEM padronizar
pca_com = PCA(n_components=2).fit(Xp)      # COM padronização

print("SEM padronizar — variância explicada:", np.round(pca_sem.explained_variance_ratio_, 3))
print("COM padronizar — variância explicada:", np.round(pca_com.explained_variance_ratio_, 3))

# Atributo dominante na PC1 em cada caso
print("\nAtributo de maior peso na PC1:")
print("  sem padronizar:", Xw.columns[np.argmax(np.abs(pca_sem.components_[0]))])
print("  com padronizar:", Xw.columns[np.argmax(np.abs(pca_com.components_[0]))])

<a id="5"></a>
# 5. Desbalanceamento de Classes — SMOTE

Quando uma classe tem muito menos exemplos que a outra, o modelo tende a
favorecer a majoritária e a **acurácia engana**.

O **SMOTE** cria amostras **sintéticas** da minoritária por interpolação:

$$x_{novo} = x_i + \lambda\,(x_{zi} - x_i), \qquad \lambda \sim U(0,1)$$

Vamos desbalancear artificialmente o **Breast Cancer** para ver o efeito.

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer(as_frame=True)
Xb, yb = bc.data, bc.target

print("Distribuição original:")
print(pd.Series(yb).value_counts().rename(index=dict(enumerate(bc.target_names))))

# Usamos um subconjunto de atributos com MAIOR SOBREPOSIÇÃO entre as classes.
# Com todos os 30 atributos o problema é fácil demais e o efeito do
# desbalanceamento fica mascarado.
feats = ["mean texture", "mean smoothness", "mean symmetry", "mean fractal dimension"]

# Desbalanceando: mantendo apenas 20 casos da classe 0 (malignant)
idx_maj = np.where(yb == 1)[0]
idx_min = np.where(yb == 0)[0][:20]
idx = np.concatenate([idx_maj, idx_min])

Xd = Xb.iloc[idx][feats].reset_index(drop=True)
yd = yb.iloc[idx].reset_index(drop=True)
print("\nApós desbalancear:")
print(yd.value_counts().rename(index=dict(enumerate(bc.target_names))))
print("Proporção da minoritária: %.1f%%" % (100 * (yd == 0).mean()))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, confusion_matrix, classification_report)

X_tr, X_te, y_tr, y_te = train_test_split(Xd, yd, test_size=0.3,
                                          stratify=yd, random_state=42)

burro = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
y_burro = burro.predict(X_te)

print("Classificador que sempre prevê a majoritária:")
print("  Acurácia : %.3f  <-- parece ótimo!" % accuracy_score(y_te, y_burro))
print("  Revocação: %.3f  <-- não detecta NENHUM caso maligno"
      % recall_score(y_te, y_burro, pos_label=0))

### Aplicando o SMOTE

Regra fundamental: **aplicar apenas no conjunto de treino**. Reamostrar antes do
split (ou reamostrar o teste) é uma forma de *data leakage* — o modelo seria
avaliado sobre amostras sintéticas derivadas do próprio treino.

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(k_neighbors=5, random_state=42)
X_bal, y_bal = sm.fit_resample(X_tr, y_tr)   # SOMENTE no treino

print("Treino antes  :", pd.Series(y_tr).value_counts().to_dict())
print("Treino depois :", pd.Series(y_bal).value_counts().to_dict())
print("Amostras sintéticas criadas:", len(y_bal) - len(y_tr))
print("\nTeste permanece INTACTO:", pd.Series(y_te).value_counts().to_dict())

### Visualizando as amostras sintéticas

Projetando em 2 componentes (PCA) para enxergar onde o SMOTE cria os pontos.

In [ ]:
pca_vis = PCA(n_components=2)
Xtr_pad = StandardScaler().fit_transform(X_tr)
Z_tr = pca_vis.fit_transform(Xtr_pad)

# Mesma transformação aplicada ao conjunto balanceado
esc_vis = StandardScaler().fit(X_tr)
Z_bal = pca_vis.transform(esc_vis.transform(X_bal))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)

ax[0].scatter(Z_tr[y_tr == 1, 0], Z_tr[y_tr == 1, 1], s=20, label="majoritária")
ax[0].scatter(Z_tr[y_tr == 0, 0], Z_tr[y_tr == 0, 1], s=30, label="minoritária")
ax[0].set_title("Treino original")

n_orig = len(y_tr)
ax[1].scatter(Z_bal[:n_orig][y_tr == 1, 0], Z_bal[:n_orig][y_tr == 1, 1],
              s=20, label="majoritária")
ax[1].scatter(Z_bal[:n_orig][y_tr == 0, 0], Z_bal[:n_orig][y_tr == 0, 1],
              s=30, label="minoritária (real)")
ax[1].scatter(Z_bal[n_orig:, 0], Z_bal[n_orig:, 1],
              s=30, marker="x", label="sintética (SMOTE)")
ax[1].set_title("Após SMOTE")

for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2")
    a.legend(fontsize=8); a.grid(False)
plt.tight_layout()
plt.show()

As amostras sintéticas (×) aparecem **entre** os pontos reais da minoritária,
sobre os segmentos que ligam vizinhos — exatamente o que a fórmula de
interpolação prevê.